# 23 拍卖指派：在线单物品


本课按百科条目写：先定义……调度台在最后，可跳过。

---

## 1. 定义

**顺序单物品拍卖**（sequential single-item auction）是一种 **在线** 指派：任务按到达序列一个个出现。每来一个任务，所有 **空闲** 机器人报一次价，价 = 从自己当前位置（本课即起点）到该任务的有向最短路；**最低者中标**，变为忙碌，不再参与后面的出价。

- **输入**：图 G，`robots`，`starts`，到达序列 `arrivals`（任务节点流）。
- **输出**：`assign[winner] = task`，以及各次中标代价之和。

它 **不是** 匈牙利。匈牙利要一次看见全部任务；拍卖在任务到达的那一刻就必须把该任务派出去，不能反悔。到达顺序不同，匹配可以不同，总价也可以不同。

历史：多机器人任务分配（MRTA）里大量用拍卖作通信协议（Gerkey、Dias、Zlot 等）。注意名字冲突：**Bertsekas 拍卖算法** 是离线解指派问题的对偶迭代，和本课「任务一个个来、最低价拿走」不是同一件事。本仓库 `auction_assign` 是后者。

---

## 2. 和谁相邻

|  | 拍卖（本课） | 匈牙利 | 贪心（22） |
|--|--------------|--------|------------|
| 看见全部任务？ | 否，在线 | 是 | 是（按车序扫） |
| 决策键 | 到达顺序 | 全局矩阵 | 机器人名单顺序 |
| 最优匹配代价 | 否 | 是 | 否 |
| 通信图像 | 每任务一次招标 | 中心算一次 | 中心按车扫 |

和 TPTS（24）：拍卖只决定谁去哪，不走路、不对头交换。TPTS 把「领最近任务」嵌进终身 MAPF。  
和 CBS-TA（25）：拍卖不管碰撞。

---

## 3. 不变量 / 定理

每来一个任务 t：

- 只有 `free` 里的车出价（忙碌车不投标）。
- 中标者 \(r^* = \arg\min_{r \in \mathrm{free}} \mathrm{Dijkstra}(s_r, t)\)（并列取名单更前）。
- 中标后 `free.remove(r*)`，该车再也不出价。

**没有**「最终匹配代价最优」不变量。只有逐步的局部规则。

**竞争比：** 相对离线最优（匈牙利）的比值。理论上可以 **任意坏**：存在 2×2 实例，在线先来「大家都便宜」的任务，把本该留给另一任务的车提前锁死，剩下的车面对巨大代价。第 6 节数字是 101 / 2.2。没有与 n 无关的常数竞争比。

实践中任务代价比较均匀时，拍卖常常接近匈牙利，所以仍常用：不等齐、实现简单、可分布式。

---

## 4. 完整伪代码（与 `auction_simple` 对应）

```
function AuctionSimple(G, robots, starts, arrivals):
    free ← copy(robots)
    assign ← {};  total ← 0
    for task in arrivals:
        if free 空: break
        bids ← []
        for r in free:
            res ← Dijkstra(G, starts[r], task)
            bid ← res.cost if res.found else 1e6
            bids.append((bid, r))
            print 本轮出价
        sort bids 升序                 # 先比价，再比名字稳定性取决于元组
        cost, winner ← bids[0]
        assign[winner] ← task
        total ← total + cost
        free.remove(winner)
    return assign, total
```

出价必须 Dijkstra，与 22 课填 C 同一套。

---

## 5. 复杂度

k 个到达、最多 n 车。第 t 个任务有 \(n-t+1\) 人出价，每人一次 Dijkstra：共 \(O(n^2)\) 次最短路，与离线填整张矩阵同阶，但 **不能** 事后再换匹配。无堆可优化「谁该中标」——人少，扫一遍即可。

---

## 6. 完整手算 / 小表

**地图上的到达顺序。** 仍用 `assign_pair`，Dijkstra 全是 2。

| 到达 | 空闲 | 出价 | 中标 | 累计 |
|------|------|------|------|------|
| G0 | r0,r1 | r0:2, r1:2 | r0（名单更前） | 2 |
| G1 | r1 | r1:2 | r1 | 4 |
| **逆序 G1** | r0,r1 | 都是 2 | r0 得 G1 | 2 |
| 然后 G0 | r1 | r1:2 | r1 得 G0 | 4 |

总价相同，**匹配对调**。在线顺序改的是谁去哪，本图碰巧钱一样。

**竞争比任意坏的 2×2**（纯数字，下面会跑）：

|  | T0 | T1 |
|--|----|----|
| r0 | 1.0 | 1.1 |
| r1 | 1.1 | 100 |

- 离线匈牙利：r0–T1（1.1）+ r1–T0（1.1）= **2.2**（另一对 1+100=101）。
- 先到 T0：r0 出 1 赢走 T0，r1 被迫接 T1=100，在线合计 **101**。
- 先到 T1：r0 出 1.1 赢走 T1，r1 接 T0=1.1，合计 **2.2**。

同一矩阵，只换到达顺序，从最优变成 40 多倍。这就是「竞争比无常数上界」的最小例子。


In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / "lib").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))


## 7. 分步实现（自己写，print traces）

先实现 `auction_simple`：每个到达打印全体出价。再在真实地图上跑 `arrivals=tasks` 与逆序，对照匈牙利离线代价。最后用数字矩阵看竞争比。


In [ ]:
from evals.scenarios import assign_pair
from lib.search import dijkstra

g, robots, starts, tasks = assign_pair()


def auction_simple(graph, robots, starts, arrivals):
    free = list(robots)
    assign = {}
    total = 0.0
    for task in arrivals:
        if not free:
            print("无空闲车，丢弃", task)
            break
        bids = []
        for r in free:
            res = dijkstra(graph, starts[r], task, record=False)
            bid = res.cost if res.found else 1e6
            bids.append((bid, r))
            print(f"  到达 {task}: {r} 出价 {bid}")
        bids.sort()
        cost, winner = bids[0]
        assign[winner] = task
        total += cost
        free.remove(winner)
        print(f"  中标 {winner}  cost={cost}  累计={total}  仍空闲 {free}")
    return assign, total


print("=== arrivals = tasks", tasks, "===")
print("结果", auction_simple(g, robots, starts, tasks))
print("=== arrivals 逆序", list(reversed(tasks)), "===")
print("结果", auction_simple(g, robots, starts, list(reversed(tasks))))


## 8. 逐行实现表

| 行 | 作用 |
|----|------|
| `free = list(robots)` | 复制。中标后从副本删，不动原名单 |
| `if not free: break` | 任务多于车时丢掉后续到达 |
| 只对 `free` 跑 Dijkstra | 忙碌车禁止出价（常见 bug 之一） |
| `bid = cost if found else 1e6` | 与 22 课不可达约定一致 |
| `bids.sort()` | 元组 (bid, name) 先比价；价同比名字 |
| `free.remove(winner)` | 中标即锁死，后面的任务看不见它 |

数字矩阵版：把 Dijkstra 换成查表，专门看顺序效应。


In [ ]:
from lib.algos.assignment import auction_assign, hungarian_assign


def auction_on_matrix(robots, tasks, C, arrivals):
    # arrivals 是任务名序列；C[i][j] 已是 r_i 对 t_j 的价
    idx = {t: j for j, t in enumerate(tasks)}
    free = list(range(len(robots)))
    assign = {}
    total = 0.0
    for task in arrivals:
        j = idx[task]
        bids = [(C[i][j], i) for i in free]
        print(f"到达 {task} 出价", [(robots[i], c) for c, i in bids])
        bids.sort()
        cost, i = bids[0]
        assign[robots[i]] = task
        total += cost
        free.remove(i)
        print(f"  中标 {robots[i]} 累计 {total}")
    return assign, total


C_online = [
    [1.0, 1.1],
    [1.1, 100.0],
]
rb, tb = ["r0", "r1"], ["T0", "T1"]
print("先 T0 再 T1（坏顺序）", auction_on_matrix(rb, tb, C_online, ["T0", "T1"]))
print("先 T1 再 T0（好顺序）", auction_on_matrix(rb, tb, C_online, ["T1", "T0"]))
print("匈牙利离线应取 1.1+1.1=2.2")

print("--- 库：地图顺序 vs 逆序 vs 匈牙利 ---")
print("auction tasks", auction_assign(g, robots, starts, tasks, arrivals=tasks))
print("auction rev  ", auction_assign(g, robots, starts, tasks, arrivals=list(reversed(tasks))))
print("hungarian    ", hungarian_assign(g, robots, starts, tasks)["cost"])


## 9. 常见 bug

1. **忙碌车继续出价。** 一人两标，后面的任务抢不到车，或一人被派两个目标。
2. **出价用欧氏距离。** 与 22 课同一类错：单行道上「直线近」的车其实绕远。
3. **把拍卖总价拿去和匈牙利比，发现更大就以为实现写错。** 在线本来就可以更差；先检查是否真的只让空闲车出价。
4. **到达里有重复任务、不从 free 删除。** 同一车中两次。
5. **和 Bertsekas 拍卖对拍。** 那是离线对偶，会收敛到最优匹配；本课算法不会。

## 10. 对照库

`lib/algos/assignment.py` 的 `auction_assign(..., arrivals=)` 即上面的循环。默认 `arrivals is None` 时用 `tasks` 原顺序。eval `test_23` 只要求指派成功、总价不低于匈牙利太多（松弛记录）。


In [ ]:
from lib.studio.widget import PlannerStudio

studio = PlannerStudio()
studio.map = g.to_studio()
studio.algo = "assignment"
studio


## 条目小结

| 项目 | 内容 |
|------|------|
| 模型 | 任务流上的单物品最低价拍卖 |
| 出价 | 空闲车的有向 Dijkstra |
| 与匈牙利 | 离线最优；在线顺序可改匹配与总价 |
| 竞争比 | 可任意坏（本课 101 vs 2.2） |
| 不是 | Bertsekas 离线拍卖 |

## 练习

1. 用 `C_online` 再手算两种到达顺序的出价表，核对 101 与 2.2。
2. 若错误地让已中标的车继续出价，先到 T0 再 T1 会变成什么匹配？
3. 改一格数字，使「坏顺序 / 匈牙利」比值超过 100。哪一格最敏感？
